## 문장의 유사도 분석
 : 두 개의 문장이 비슷한 것인지 혹은 관련이 있는 것인지 분석

#### 코사인(Cosine) 유사도

In [ ]:
a='딥러닝은 매우 재미있는 기술이라 공부하고 있습니다.'
# b='공부하면 재미있는 기술이라 딥러닝을 배우고 있습니다.' # 코사인 유사도 측정 :  [[0.33609693]]
# b='동해물과 백두산이 마르고닳도록 하느님이 보우하사 우리나라만세' # 코사인 유사도 측정 :  [[0.]]
b='딥러닝은 매우 재미있는 기술이라 공부하고 있습니다.' # 코사인 유사도 측정 :  [[1.]]

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer     # feature 에서 텍스트를 추출하는 애

In [17]:
sentences = (a, b)
tfid_vectorizer = TfidfVectorizer()

In [18]:
sentences

('딥러닝은 매우 재미있는 기술이라 공부하고 있습니다.', '딥러닝은 매우 재미있는 기술이라 공부하고 있습니다.')

In [19]:
# 문장 벡터화 하기 : 사전 만들기
tfid_matrix = tfid_vectorizer.fit_transform(sentences)

In [20]:
tfid_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (2, 6)>

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
# 첫 번째 문장과 두 번째 문장을 비교
cos_similar = cosine_similarity(tfid_matrix[0:1], tfid_matrix[1:2])
print("코사인 유사도 측정 : ", cos_similar)

코사인 유사도 측정 :  [[1.]]


----
### 유사도를 이용한 추천 시스템 구현
 : 코사인 유사도만으로 영화의 줄거리에 기반하여 영화를 추천하는 시스템

In [22]:
import pandas as pd

In [23]:
data = pd.read_csv('../Data/movies_metadata/movies_metadata.csv', low_memory=False)
data.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [24]:
data.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [25]:
data[['title', 'overview']].head()

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [26]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [27]:
# 상위 2만 개만 쓰자
data = data.head(20000)

In [28]:
data['overview'].isnull().sum()

np.int64(135)

In [29]:
# 결측치를 빈 값으로 대체
data['overview'] = data['overview'].fillna('')

In [30]:
data['overview'].isnull().sum()

np.int64(0)

In [31]:
# 행렬 크기 구하기
tfidf = TfidfVectorizer(stop_words='english')   # 불용어. 이 경우엔 영어 제외하고 안 씀
tfid_matrix = tfidf.fit_transform(data['overview'])
tfid_matrix.shape

(20000, 47487)

In [32]:
cosine_sim = cosine_similarity(tfid_matrix, tfid_matrix)
cosine_sim.shape

(20000, 20000)

In [33]:
cosine_sim

array([[1.        , 0.01575748, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.01575748, 1.        , 0.04907345, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.04907345, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.08375766],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.08375766, 0.        ,
        1.        ]], shape=(20000, 20000))

In [34]:
# 기존 데이터 프레임으로부터 영화 타이틀을 key, 영화 index 를 value 로 하는 딕셔너리를 만들어보자
title_to_index = dict(zip(data['title'], data.index))

In [36]:
# 확인 작업
title_to_index['Father of the Bride Part II']

4

In [ ]:
# 선택한 영화의 제목을 입력하면 코사인 유사도를 통해 가장 overview 가 유사한 10개의 영화를 찾아내는 함수

def get_recommendation(title, cosine_sim = cosine_sim) : # 나중에 바꿀까봐 일단 cosine_sim 기본값 부여
    # 선택한 영화의 타이틀로부터 해당 영화의 인덱스를 받아온다.
    idx = title_to_index[title]
    # 해당 영화와 모든 영화와의 유사도를 가져온다.
    sim_scores = list(enumerate(cosine_sim[idx]))
    # 유사도에 따라 영화들을 정렬한다.
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)   # 유사도 높은 걸 위에 두기 위해 리버스
    # 가장 유사한 10개의 영화를 받아 온다.
    sim_scores = sim_scores[1:11]
    # 가장 유사한 10개 영화의 인덱스를 가져온다.
    movies_indices = [idx[0] for idx in sim_scores]
    # 가져온 영화의 제목을 리턴한다.
    return data['title'].iloc[movies_indices]

In [40]:
# 영화 제목 입력하면 10개 추천해줌.
get_recommendation('The Dark Knight Rises')

12481                            The Dark Knight
150                               Batman Forever
1328                              Batman Returns
15511                 Batman: Under the Red Hood
585                                       Batman
9230          Batman Beyond: Return of the Joker
18035                           Batman: Year One
19792    Batman: The Dark Knight Returns, Part 1
3095                Batman: Mask of the Phantasm
10122                              Batman Begins
Name: title, dtype: object